In [2]:
import os
import pandas as pd
import numpy as np
import gc

df = pd.read_parquet("../processed/cleaned.parquet")  # Reading the cleaned data

## TASK 1 has_identity_data 

This is the most important feature you will build today. It captures Vesta's selection bias: they only
computed expensive device fingerprints for transactions that already triggered risk indicators. So the
presence of identity data is itself a pre-screening signal.

Create a new column has_identity_data that is 1 when identity data is available for a transaction, and 0
when it is not

In [3]:
df["has_identity"] = df["DeviceType"].notna().astype(np.int8)  # Creating a new feature to indicate if identity information is present
print(df[["has_identity","DeviceType"]].head(20))



    has_identity DeviceType
0              0       None
1              0       None
2              0       None
3              0       None
4              1     mobile
5              0       None
6              0       None
7              0       None
8              1     mobile
9              0       None
10             1    desktop
11             1    desktop
12             0       None
13             0       None
14             0       None
15             0       None
16             1    desktop
17             1    desktop
18             0       None
19             0       None


In [4]:
df["has_identity"].value_counts()

has_identity
0    449730
1    140810
Name: count, dtype: int64

In [8]:
df.groupby("has_identity")["isFraud"].mean() * 100

has_identity
0    2.101705
1    7.961792
Name: isFraud, dtype: float64

In [10]:
(df.groupby("has_identity")["isFraud"].mean() * 100).round(2).astype(str) + "%"

has_identity
0     2.1%
1    7.96%
Name: isFraud, dtype: object

## TASK 2-7 Six Missingness Flags

In [ ]:
flags = {
    "has_R_emaildomain": "R_emaildomain",
    "has_P_emaildomain": "P_emaildomain",
    "has_card2": "card2",
    "has_card3": "card3",
    "has_addr1": "addr1",
    "has_dist1": "dist1"
}

for new_col, source_col in flags.items():
    df[new_col] = df[source_col].notna().astype("int8")

    

In [12]:
print(df[list(flags.keys())].head())

   has_R_emaildomain  has_P_emaildomain  has_card2  has_card3  has_addr1  \
0                  0                  0          0          1          1   
1                  0                  1          1          1          1   
2                  0                  1          1          1          1   
3                  0                  1          1          1          1   
4                  0                  1          1          1          1   

   has_dist1  
0          1  
1          0  
2          1  
3          0  
4          0  


In [ ]:
for col in flags.keys():
    print("\n", col)
##     print(df.groupby(col)["isFraud"].mean() * 100)


 has_R_emaildomain
has_R_emaildomain
0    2.081858
1    8.177521
Name: isFraud, dtype: float64

 has_P_emaildomain
has_P_emaildomain
0    2.953756
1    3.602817
Name: isFraud, dtype: float64

 has_card2
has_card2
0    4.735251
1    3.480013
Name: isFraud, dtype: float64

 has_card3
has_card3
0    2.492013
1    3.501677
Name: isFraud, dtype: float64

 has_addr1
has_addr1
0    11.781268
1     2.462112
Name: isFraud, dtype: float64

 has_dist1
has_dist1
0    4.515841
1    1.995644
Name: isFraud, dtype: float64


## TASK 8 log_amount


In [14]:
df["log_amount"] = np.log1p(df["TransactionAmt"]).astype("float32")

In [15]:
print(df[["TransactionAmt", "log_amount"]].describe())

       TransactionAmt     log_amount
count   590540.000000  590540.000000
mean       135.027176       4.382960
std        239.162522       0.937183
min          0.251000       0.223943
25%         43.321000       3.791459
50%         68.769000       4.245190
75%        125.000000       4.836282
max      31937.391000      10.371564


## TASK 9-10 Email Domain Frequency Encoding


In [16]:
df["P_emaildomain_freq"] = (
    df["P_emaildomain"]
    .map(df["P_emaildomain"].value_counts().to_dict())
    .fillna(0)
    .astype("int32")
)

df["R_emaildomain_freq"] = (
    df["R_emaildomain"]
    .map(df["R_emaildomain"].value_counts().to_dict())
    .fillna(0)
    .astype("int32")
)

In [17]:
print(
    df[
        ["P_emaildomain", "P_emaildomain_freq",
         "R_emaildomain", "R_emaildomain_freq"]
    ].head(10)
)

   P_emaildomain  P_emaildomain_freq R_emaildomain  R_emaildomain_freq
0            NaN                   0           NaN                   0
1      gmail.com              228355           NaN                   0
2    outlook.com                5096           NaN                   0
3      yahoo.com              100934           NaN                   0
4      gmail.com              228355           NaN                   0
5      gmail.com              228355           NaN                   0
6      yahoo.com              100934           NaN                   0
7       mail.com                 559           NaN                   0
8  anonymous.com               36998           NaN                   0
9      yahoo.com              100934           NaN                   0


In [18]:
print(df["P_emaildomain"].value_counts().head(10))

P_emaildomain
gmail.com        228355
yahoo.com        100934
hotmail.com       45250
anonymous.com     36998
aol.com           28289
comcast.net        7888
icloud.com         6267
outlook.com        5096
msn.com            4092
att.net            4033
Name: count, dtype: int64


## TASK 11 DeviceInfo Frequency Encoding


In [19]:
if "DeviceInfo" in df.columns:
    
    df["DeviceInfo_freq"] = (
        df["DeviceInfo"]
        .map(df["DeviceInfo"].value_counts().to_dict())
        .fillna(0)
        .astype("int32")
    )

    print(df["DeviceInfo_freq"].min())
    print(df["DeviceInfo_freq"].max())

0
47722


## TASK 12 is_mobile


In [20]:
df["is_mobile"] = (df["DeviceType"] == "mobile").astype("int8")

In [21]:
df[df["DeviceType"].notna()].groupby("is_mobile")["isFraud"].mean() * 100

is_mobile
0     6.521458
1    10.166232
Name: isFraud, dtype: float64

## TASK 13 amount_bucket


In [25]:
def amount_bucket(x):
    if x < 10:
        return "micro"
    elif x < 50:
        return "small"
    elif x < 200:
        return "medium"
    elif x < 1000:
        return "large"
    else:
        return "very_large"

df["amount_bucket"] = df["TransactionAmt"].apply(amount_bucket)

In [26]:
df.groupby("amount_bucket")["isFraud"].mean() * 100

amount_bucket
large         4.714178
medium        2.848077
micro         8.329654
small         3.825808
very_large    2.467018
Name: isFraud, dtype: float64